# Near-Misses — edit-recovery probe with Gemma 4 (small-scale, real)

Companion to the Kaggle Paper Track writeup *Near-Misses, Not Wrong Turns*. Repo: https://github.com/San-Lins/swe-failure-taxonomy

**What this measures.** For each of the 250 annotated GPT-4o failures we find the *first failed edit* (an OpenHands `str_replace_editor` call whose observation is an explicit tool error, e.g. `old_str ... did not appear verbatim`). We rebuild the exact file state at that moment by replaying every earlier editor command on top of the repository at `base_commit`, check that the failure reproduces, and then give a recovery agent the issue, the failed call and its error, plus `view` / `str_replace` tools on that reconstructed file system for at most 6 steps.

**Outcomes (no test execution):** the agent *recovers* if it lands an edit that applies cleanly and differs from the failed one; we also record whether the edited Python file still parses and whether the edit is in a file touched by the gold patch. The same outcomes are computed for what GPT-4o itself did in the next 6 turns of the original trajectory, as a baseline.

Settings: GPU T4 ×2, Internet on.

In [ ]:
%%capture
!pip install -q -U "transformers>=4.57" accelerate bitsandbytes datasets kagglehub

In [ ]:
import kagglehub
for h in ["google/gemma-4/transformers/gemma-4-e4b-it", "google/gemma-4/transformers/gemma-4-12b-it"]:
    try: print(h, "->", kagglehub.model_download(h))
    except Exception as e: print("could not attach", h, repr(e)[:200])

In [ ]:
import os, re, json, time, subprocess, ast, collections, math, sys
import pandas as pd
WORK = "/kaggle/working"
REPO = f"{WORK}/swe-failure-taxonomy"
if not os.path.exists(REPO):
    subprocess.run(["git","clone","--depth","1","https://github.com/San-Lins/swe-failure-taxonomy",REPO],check=True)
labels = json.load(open(f"{REPO}/data/full_labels_v3.json"))
sys.path.insert(0, f"{REPO}/eval-proto/edit-recovery")
from truncate import FAIL_PATTERNS, EDIT_COMMANDS   # failure signals mined from real trajectories

from datasets import load_dataset, get_dataset_split_names
TRAJ = "SWE-Gym/OpenHands-Sampled-Trajectories"
traj = load_dataset(TRAJ, split=get_dataset_split_names(TRAJ)[0])
gym = {r["instance_id"]: r for r in load_dataset("SWE-Gym/SWE-Gym", split="train")}
rows = []
for L in labels:
    r = traj[int(L["idx"])]
    assert r["instance_id"] == L["instance_id"]
    rows.append((L, r))
print(len(rows), "joined")

## 1. Find the first failed edit and rebuild file state by replay

In [ ]:
def text_of(c):
    if c is None: return ""
    if isinstance(c, str): return c
    if isinstance(c, list): return "".join(x.get("text","") if isinstance(x, dict) else str(x) for x in c)
    return str(c)

def repo_dir_of(msgs):
    for m in msgs:
        if m.get("role") == "user":
            mm = re.search(r"<uploaded_files>\s*(/workspace/\S+)", text_of(m.get("content")))
            if mm: return mm.group(1).rstrip("/")
    return None

def is_fail(obs): return any(p.search(obs) for p in FAIL_PATTERNS) or obs.lstrip().startswith("ERROR")

def editor_calls(msgs):
    # yield (msg_index, args, observation) for every str_replace_editor call
    for i, m in enumerate(msgs):
        for tc in (m.get("tool_calls") or []):
            fn = tc.get("function", {})
            if fn.get("name") != "str_replace_editor": continue
            try: args = json.loads(fn.get("arguments") or "{}")
            except Exception: continue
            obs = ""
            for n in msgs[i+1:i+4]:
                if n.get("role") == "tool" and (not tc.get("id") or n.get("tool_call_id") in (None, tc.get("id"))):
                    obs = text_of(n.get("content")); break
            yield i, args, obs

GIT = {}
def repo_clone(repo):
    d = f"/kaggle/temp/repos/{repo.replace('/','__')}"
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True)
        subprocess.run(["git","clone","--filter=blob:none","--no-checkout","-q",f"https://github.com/{repo}.git",d],check=True)
    return d
def git_show(repo, commit, rel):
    d = repo_clone(repo)
    out = subprocess.run(["git","-C",d,"show",f"{commit}:{rel}"],capture_output=True)
    return out.stdout.decode("utf-8","replace") if out.returncode == 0 else None

class FS:
    # in-memory str_replace_editor over the repo at base_commit
    def __init__(self, repo, commit, root):
        self.repo, self.commit, self.root = repo, commit, root
        self.files, self.hist = {}, collections.defaultdict(list)
    def rel(self, path):
        if path and path.startswith(self.root + "/"): return path[len(self.root)+1:]
        return None
    def get(self, path):
        if path not in self.files:
            r = self.rel(path)
            self.files[path] = git_show(self.repo, self.commit, r) if r else None
        return self.files[path]
    def apply(self, args):
        # returns (ok, message); mirrors OpenHands semantics closely enough to replay
        cmd, path = args.get("command"), args.get("path")
        if cmd == "view": return True, "view"
        if cmd == "create":
            if self.get(path) is not None: return False, "ERROR: file already exists"
            self.hist[path].append(None); self.files[path] = args.get("file_text") or ""; return True, "created"
        cur = self.get(path)
        if cur is None: return False, "ERROR: path does not exist"
        if cmd == "str_replace":
            old, new = args.get("old_str"), args.get("new_str") or ""
            if old is None: return False, "ERROR: old_str missing"
            n = cur.count(old)
            if n == 0: return False, f"ERROR: No replacement was performed, old_str did not appear verbatim in {path}."
            if n > 1: return False, f"ERROR: No replacement was performed. Multiple occurrences of old_str in {path}."
            self.hist[path].append(cur); self.files[path] = cur.replace(old, new, 1); return True, "edited"
        if cmd == "insert":
            try: k = int(args.get("insert_line"))
            except Exception: return False, "ERROR: bad insert_line"
            lines = cur.split("\n")
            if k < 0 or k > len(lines): return False, "ERROR: insert_line out of range"
            self.hist[path].append(cur); lines[k:k] = (args.get("new_str") or "").split("\n")
            self.files[path] = "\n".join(lines); return True, "inserted"
        if cmd == "undo_edit":
            if not self.hist[path]: return False, "ERROR: no edit history"
            self.files[path] = self.hist[path].pop(); return True, "undone"
        return False, f"ERROR: unknown command {cmd}"

def find_case(L, r):
    msgs = r["messages"]; root = repo_dir_of(msgs); g = gym.get(L["instance_id"])
    if not root or not g: return None, "no_root_or_gold"
    fs = FS(g["repo"], g["base_commit"], root)
    calls = list(editor_calls(msgs)); diverged = False
    for j, (i, args, obs) in enumerate(calls):
        if args.get("command") in EDIT_COMMANDS and is_fail(obs) and any(p.search(obs) for p in FAIL_PATTERNS):
            ok, _ = fs.apply(dict(args))            # replay the failing call on our reconstruction
            reproduced = not ok
            if ok: fs.apply({"command":"undo_edit","path":args.get("path")})
            return dict(L=L, r=r, fs=fs, root=root, fail_idx=i, fail_args=args, fail_obs=obs[:600],
                        later=calls[j+1:], reproduced=reproduced and not diverged, diverged=diverged), "truncatable"
        if args.get("command") != "view" and not is_fail(obs):
            ok2, _ = fs.apply(dict(args))             # successful earlier edit -> replay it
            if not ok2 and args.get("command") in EDIT_COMMANDS | {"undo_edit"}: diverged = True
    return None, "no_failed_edit"

CASES, status = [], collections.Counter()
for L, r in rows:
    c, s = find_case(L, r); status[s] += 1
    if c: CASES.append(c)
repro = [c for c in CASES if c["reproduced"]]
print(status, "| failure reproduces on reconstruction:", len(repro), "/", len(CASES), "| diverged replays:", sum(c["diverged"] for c in CASES))
print(collections.Counter(c["L"]["primary_v3"] for c in repro))

## 2. GPT-4o baseline: what the original agent did in its next 6 turns

In [ ]:
MAX_STEPS = 6
GOLD_RE = re.compile(r"^diff --git a/(.*?) b/", re.M)
def gold_files(iid): return set(GOLD_RE.findall(gym[iid]["patch"]))
def parses(path, content):
    if not path.endswith(".py") or content is None: return None
    try: ast.parse(content); return True
    except Exception: return False

def gpt4o_baseline(c):
    import copy
    fs = copy.deepcopy(c["fs"]); fail = c["fail_args"]
    turns = sorted({i for i, _, _ in c["later"]})
    window = set(turns[:MAX_STEPS])
    for i, args, obs in c["later"]:
        if i not in window: break
        if args.get("command") in EDIT_COMMANDS:
            same = (args.get("old_str") == fail.get("old_str") and args.get("new_str") == fail.get("new_str") and args.get("path") == fail.get("path"))
            ok, _ = fs.apply(dict(args))
            if ok and not same:
                p = args.get("path"); rel = c["fs"].rel(p) or ""
                return dict(recovered=True, steps=turns.index(i)+1, parses=parses(p, fs.files.get(p)),
                            in_gold=rel in gold_files(c["L"]["instance_id"]), repeated_identical=False)
        elif args.get("command") == "undo_edit":
            fs.apply(dict(args))
    rep = any(a.get("old_str") == fail.get("old_str") and a.get("path") == fail.get("path") for i, a, _ in c["later"] if i in window)
    return dict(recovered=False, steps=None, parses=None, in_gold=False, repeated_identical=rep)

BASE = {c["L"]["instance_id"]+"#"+str(c["L"]["idx"]): gpt4o_baseline(c) for c in repro}
b = pd.DataFrame(BASE.values()); print(b.recovered.mean(), b.repeated_identical.mean())

## 3. Gemma 4 recovery agent (view / str_replace tools, ≤ 6 steps)

In [ ]:
import torch, gc, copy
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
HANDLES = {"gemma-4-e4b-it": "google/gemma-4/transformers/gemma-4-e4b-it",
           "gemma-4-12b-it": "google/gemma-4/transformers/gemma-4-12b-it"}
def load(name):
    path = kagglehub.model_download(HANDLES[name])
    proc = AutoProcessor.from_pretrained(path)
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(path, quantization_config=q, device_map="auto", dtype=torch.float16).eval()
    return proc, model

SYS = ("You are a software engineering agent repairing a failed file edit. Respond with exactly one JSON object per turn, "
       "one of: {\"tool\":\"view\",\"path\":P,\"start\":S,\"end\":E} | "
       "{\"tool\":\"str_replace\",\"path\":P,\"old_str\":O,\"new_str\":N} | {\"tool\":\"finish\"}. "
       "old_str must match the file exactly once, including indentation. No other text.")

def first_prompt(c):
    f = c["fail_args"]; iid = c["L"]["instance_id"]
    return (f"Repository root: {c['root']}\n\nIssue:\n{gym[iid]['problem_statement'][:3000]}\n\n"
            f"Your previous edit FAILED and was not applied.\ncommand: {f.get('command')}\npath: {f.get('path')}\n"
            f"old_str:\n{(f.get('old_str') or '')[:1500]}\nnew_str:\n{(f.get('new_str') or '')[:1500]}\n"
            f"error: {c['fail_obs'][:400]}\n\nDiagnose why it failed (view the file if needed) and make the intended change. "
            "Do not repeat the identical edit.")

def view(fs, args):
    cur = fs.get(args.get("path"))
    if cur is None: return "ERROR: path does not exist"
    lines = cur.split("\n")
    try: s = max(1, int(args.get("start") or 1)); e = min(len(lines), int(args.get("end") or s+80), s+79)
    except Exception: s, e = 1, min(len(lines), 80)
    return "\n".join(f"{k:6d}\t{lines[k-1]}" for k in range(s, e+1))

def parse_action(txt):
    m = re.search(r"\{.*\}", txt, re.S)
    if not m: return None
    try: return json.loads(m.group(0))
    except Exception:
        try: return json.loads(m.group(0).replace("\n", "\\n"))
        except Exception: return None

def gen(proc, model, msgs, max_new=700):
    text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    enc = proc(text=text, return_tensors="pt").to(model.device)
    if enc["input_ids"].shape[-1] > 14000: return '{"tool":"finish"}'
    with torch.no_grad(): out = model.generate(**enc, max_new_tokens=max_new, do_sample=False)
    return proc.decode(out[0][enc["input_ids"].shape[-1]:], skip_special_tokens=True)

def run_case(proc, model, c):
    fs = copy.deepcopy(c["fs"]); f = c["fail_args"]
    msgs = [{"role":"system","content":SYS},{"role":"user","content":first_prompt(c)}]
    tool_errors = identical = 0
    for step in range(1, MAX_STEPS+1):
        txt = gen(proc, model, msgs); msgs.append({"role":"assistant","content":txt})
        a = parse_action(txt)
        if not a or a.get("tool") == "finish": obs = "ERROR: invalid or no action" if not a else None
        if a and a.get("tool") == "finish": break
        if a and a.get("tool") == "view": obs = view(fs, a)
        elif a and a.get("tool") == "str_replace":
            args = {"command":"str_replace","path":a.get("path"),"old_str":a.get("old_str"),"new_str":a.get("new_str")}
            same = a.get("old_str") == f.get("old_str") and a.get("new_str") == f.get("new_str") and a.get("path") == f.get("path")
            identical += same
            ok, obs = fs.apply(args)
            if ok and not same:
                p = a.get("path"); rel = c["fs"].rel(p) or ""
                return dict(recovered=True, steps=step, parses=parses(p, fs.files.get(p)), in_gold=rel in gold_files(c["L"]["instance_id"]),
                            tool_errors=tool_errors, identical=identical)
            tool_errors += (not ok)
        elif a: obs = "ERROR: unknown tool"
        msgs.append({"role":"user","content":"Observation:\n" + (obs or "")[:4000]})
    return dict(recovered=False, steps=None, parses=None, in_gold=False, tool_errors=tool_errors, identical=identical)

RES = {}
for name in ["gemma-4-12b-it", "gemma-4-e4b-it"]:
    out = f"{WORK}/er_{name}.jsonl"; done = {}
    if os.path.exists(out):
        for l in open(out): d = json.loads(l); done[d["key"]] = d
    try:
        proc, model = load(name); t0 = time.time()
        with open(out, "a") as fo:
            for k, c in enumerate(repro):
                key = c["L"]["instance_id"]+"#"+str(c["L"]["idx"])
                if key in done: continue
                try: d = run_case(proc, model, c)
                except Exception as e: d = dict(recovered=False, steps=None, parses=None, in_gold=False, tool_errors=0, identical=0, error=repr(e)[:200])
                d["key"] = key; done[key] = d; fo.write(json.dumps(d)+"\n"); fo.flush()
                if k % 10 == 0: print(name, k, "/", len(repro), f"{time.time()-t0:.0f}s")
        del model; gc.collect(); torch.cuda.empty_cache()
    except Exception as e:
        print("FAILED", name, repr(e)[:300])
    RES[name] = done

## 4. Results

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return (float('nan'),)*2
    p=k/n; d=1+z*z/n; c=(p+z*z/(2*n))/d; h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return round(100*(c-h),1), round(100*(c+h),1)
def summarize(name, recs):
    recs = list(recs); n = len(recs); k = sum(r["recovered"] for r in recs)
    rec = [r for r in recs if r["recovered"]]
    py = [r for r in rec if r["parses"] is not None]
    return dict(system=name, n=n, recovered=k, recovery_pct=round(100*k/n,1) if n else None, ci=wilson(k,n),
                median_steps=(sorted(r["steps"] for r in rec)[len(rec)//2] if rec else None),
                parses_pct=round(100*sum(r["parses"] for r in py)/len(py),1) if py else None,
                in_gold_pct=round(100*sum(r["in_gold"] for r in rec)/len(rec),1) if rec else None)
keys = [c["L"]["instance_id"]+"#"+str(c["L"]["idx"]) for c in repro]
table = [summarize("GPT-4o (original trajectory, next 6 turns)", [BASE[k] for k in keys])]
for name, done in RES.items():
    table.append(summarize(name, [done[k] for k in keys if k in done]))
T = pd.DataFrame(table); print(T.to_string(index=False))
by = collections.defaultdict(dict)
for k, c in zip(keys, repro):
    br = "LOC" if c["L"]["primary_v3"].startswith("LOC") else ("NARROW" if c["L"]["primary_v3"].startswith("NARROW") else "other")
    by[br].setdefault("n", 0); by[br]["n"] += 1
    by[br]["gpt4o"] = by[br].get("gpt4o", 0) + BASE[k]["recovered"]
    for name, done in RES.items():
        if k in done: by[br][name] = by[br].get(name, 0) + done[k]["recovered"]
print(json.dumps(by, indent=1))
summary = dict(status=dict(status), n_truncatable=len(CASES), n_reproduced=len(repro), n_labels=len(rows),
               truncation_yield=round(len(CASES)/len(rows),3), table=table, by_branch=by,
               gpt4o_repeated_identical=round(float(b.repeated_identical.mean()),3))
json.dump(summary, open(f"{WORK}/edit_recovery_summary.json","w"), indent=1, default=str)
print(json.dumps(summary, indent=1, default=str))